In [1]:
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

plt.style.use('seaborn-v0_8')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 🔧 Model Definitions

In [3]:
class RNN(nn.Module):
    """Vanilla RNN - Current best architecture"""
    def __init__(self, input_size, output_size=1, rnn_units=64, fc_units=64, 
                 activation_fct=nn.ReLU(), dropout=[0.0, 0, 0], rnn_nonlinearity='relu'):
        super(RNN, self).__init__()
        self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
        self.fc = nn.Linear(rnn_units, fc_units)
        self.output_layer = nn.Linear(fc_units, output_size)
        self.dropout1 = nn.Dropout(dropout[0])
        self.dropout2 = nn.Dropout(dropout[1])
        self.dropout3 = nn.Dropout(dropout[2])
        self.activation_fct = activation_fct
 
    def forward(self, x):
        x = self.dropout1(x)
        x, _ = self.rnn_layer(x)
        x = x[:, -1, :]
        x = self.dropout2(x)
        x = self.activation_fct(self.fc(x))
        x = self.dropout3(x)
        x = self.output_layer(x)
        return x







## 📊 Load Data

In [4]:
print("Loading data...")
train_dataloader, val_dataloader, pipeline = load_data(
    train_path = "../datasets_challenge/train",
    train_ratio = 0.98,
    batchsize = 64,
    columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],
    sequence_length = 20,
    threshold = True,
    threshold_value = 3.0,
    threshold_column = "vxCG",
    target = "sideSlip",
    scaling = True,
    seed = 42,
)

X_sample, y_sample = next(iter(train_dataloader))
input_size = X_sample.shape[2]

print(f"✅ Data loaded")
print(f"   Input size: {input_size}")
print(f"   Train batches: {len(train_dataloader)}")
print(f"   Val batches: {len(val_dataloader)}")

Loading data...
../datasets_challenge/train\train_part_001.csv
../datasets_challenge/train\train_part_002.csv
../datasets_challenge/train\train_part_003.csv
../datasets_challenge/train\train_part_004.csv
../datasets_challenge/train\train_part_005.csv
../datasets_challenge/train\train_part_006.csv
../datasets_challenge/train\train_part_007.csv
../datasets_challenge/train\train_part_008.csv
../datasets_challenge/train\train_part_009.csv
../datasets_challenge/train\train_part_005.csv
../datasets_challenge/train\train_part_006.csv
../datasets_challenge/train\train_part_007.csv
../datasets_challenge/train\train_part_008.csv
../datasets_challenge/train\train_part_009.csv
../datasets_challenge/train\train_part_010.csv
../datasets_challenge/train\train_part_011.csv
../datasets_challenge/train\train_part_012.csv
../datasets_challenge/train\train_part_013.csv
../datasets_challenge/train\train_part_010.csv
../datasets_challenge/train\train_part_011.csv
../datasets_challenge/train\train_part_012.c

##  Strategy 1: Multi-Seed Ensemble +

Train the **exact same architecture** with 5 different seeds and average predictions.



In [5]:
def train_single_model(model, train_loader, val_loader, criterion, optimizer, 
                       num_epochs=50, early_stop_thresh=10, device='cuda', verbose=True):
    """Train a single model and return best checkpoint"""
    best_loss = np.inf
    best_epoch = -1
    best_model_state = None
    train_curve = []
    val_curve = []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        running_loss = 0
        for x, y in train_loader:
            X, y = x.to(device), y.to(device).reshape([len(y), 1])
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            running_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        avg_train_loss = running_loss / len(train_loader)
        train_curve.append(avg_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                X_val, y_val = x.to(device), y.to(device).reshape([len(y), 1])
                val_loss = criterion(model(X_val), y_val)
                running_val_loss += val_loss.item()
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_curve.append(avg_val_loss)
        
        if verbose:
            print(f"Epoch {epoch+1:2d}/{num_epochs} | Train: {avg_train_loss:.5f} | Val: {avg_val_loss:.5f}", end="")
        
        # Early stopping
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_epoch = epoch
            best_model_state = model.state_dict().copy()
            if verbose:
                print(" 💾")
        elif epoch - best_epoch > early_stop_thresh:
            if verbose:
                print(f"\n⏹️  Early stop at epoch {epoch+1}")
            break
        else:
            if verbose:
                print("")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, best_loss, train_curve, val_curve

In [6]:
def predict_with_model(model, test_loader, device='cuda'):
    """Generate predictions with a trained model"""
    model.eval()
    predictions = []
    with torch.no_grad():
        for x, _ in test_loader:
            X = x.to(device)
            pred = model(X)
            predictions.append(pred.cpu().numpy())
    return np.concatenate(predictions).flatten()

In [7]:
# Train 5 models with different seeds
SEEDS = [42, 123, 456, 789, 2024]
NUM_MODELS = len(SEEDS)

models = []
val_losses = []

print("\n" + "="*80)
print(f"🚀 TRAINING {NUM_MODELS} MODELS WITH DIFFERENT SEEDS")
print("="*80 + "\n")

for i, seed in enumerate(SEEDS):
    print(f"\n{'='*80}")
    print(f"📦 MODEL {i+1}/{NUM_MODELS} | SEED = {seed}")
    print("="*80)
    
    # Set seed
    seed_everything(seed)
    
    # Create model
    model = RNN(
        input_size=input_size,
        output_size=1,
        rnn_units=64,
        fc_units=64,
        dropout=[0.0, 0.0, 0.0]
    ).to(device)
    
    # Train
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    model, val_loss, _, _ = train_single_model(
        model, train_dataloader, val_dataloader, criterion, optimizer,
        num_epochs=70, early_stop_thresh=10, device=device, verbose=True
    )
    
    models.append(model)
    val_losses.append(val_loss)
    
    print(f"✅ Model {i+1} trained | Val Loss: {val_loss:.5f}")

print("\n" + "="*80)
print("✅ ALL MODELS TRAINED")
print("="*80)
print(f"Val losses: {[f'{v:.5f}' for v in val_losses]}")
print(f"Mean val loss: {np.mean(val_losses):.5f}")
print(f"Std val loss:  {np.std(val_losses):.5f}")


🚀 TRAINING 5 MODELS WITH DIFFERENT SEEDS


📦 MODEL 1/5 | SEED = 42
Epoch  1/70 | Train: 0.27474 | Val: 0.19438 💾
Epoch  1/70 | Train: 0.27474 | Val: 0.19438 💾
Epoch  2/70 | Train: 0.19106 | Val: 0.17790 💾
Epoch  2/70 | Train: 0.19106 | Val: 0.17790 💾
Epoch  3/70 | Train: 0.17605 | Val: 0.16621 💾
Epoch  3/70 | Train: 0.17605 | Val: 0.16621 💾
Epoch  4/70 | Train: 0.16763 | Val: 0.15767 💾
Epoch  4/70 | Train: 0.16763 | Val: 0.15767 💾
Epoch  5/70 | Train: 0.16324 | Val: 0.15900
Epoch  5/70 | Train: 0.16324 | Val: 0.15900
Epoch  6/70 | Train: 0.15837 | Val: 0.15631 💾
Epoch  6/70 | Train: 0.15837 | Val: 0.15631 💾
Epoch  7/70 | Train: 0.15522 | Val: 0.15747
Epoch  7/70 | Train: 0.15522 | Val: 0.15747
Epoch  8/70 | Train: 0.15264 | Val: 0.15926
Epoch  8/70 | Train: 0.15264 | Val: 0.15926
Epoch  9/70 | Train: 0.15056 | Val: 0.15863
Epoch  9/70 | Train: 0.15056 | Val: 0.15863
Epoch 10/70 | Train: 0.14821 | Val: 0.14501 💾
Epoch 10/70 | Train: 0.14821 | Val: 0.14501 💾
Epoch 11/70 | Train: 0.14681

In [11]:
# Load test data
print("\nLoading test data...")
test_dataloader = load_test_data(
    test_path = "../datasets_challenge/test",
    batchsize = 512,
    pipeline = pipeline,
    load_anonymized_data=True
)
print("✅ Test data loaded")


Loading test data...
../datasets_challenge/test\test_part_001.csv
../datasets_challenge/test\test_part_002.csv
../datasets_challenge/test\test_part_003.csv
../datasets_challenge/test\test_part_004.csv
../datasets_challenge/test\test_part_005.csv
../datasets_challenge/test\test_part_006.csv
../datasets_challenge/test\test_part_007.csv
../datasets_challenge/test\test_part_008.csv
../datasets_challenge/test\test_part_009.csv
../datasets_challenge/test\test_part_010.csv
../datasets_challenge/test\test_part_011.csv
✅ Test data loaded


In [12]:
# Generate predictions from all models
print("\n" + "="*80)
print("🔮 GENERATING ENSEMBLE PREDICTIONS")
print("="*80)

all_predictions = []
for i, model in enumerate(models):
    print(f"Predicting with model {i+1}/{NUM_MODELS}...", end=" ")
    preds = predict_with_model(model, test_dataloader, device)
    all_predictions.append(preds)
    print(f"✅ Range: [{preds.min():.3f}, {preds.max():.3f}]")

# Simple average ensemble
ensemble_preds = np.mean(all_predictions, axis=0)

print("\n" + "="*80)
print("📊 ENSEMBLE STATISTICS")
print("="*80)
print(f"Num models: {NUM_MODELS}")
print(f"Ensemble range: [{ensemble_preds.min():.3f}, {ensemble_preds.max():.3f}]")
print(f"Ensemble mean:  {ensemble_preds.mean():.3f}")
print(f"Ensemble std:   {ensemble_preds.std():.3f}")

# Individual model statistics
print("\nIndividual model std devs:")
for i, preds in enumerate(all_predictions):
    print(f"  Model {i+1}: std={preds.std():.3f}")

# Prediction variance across models
pred_variance = np.var(all_predictions, axis=0)
print(f"\nPrediction variance (mean): {pred_variance.mean():.6f}")
print(f"Prediction variance (max):  {pred_variance.max():.6f}")


🔮 GENERATING ENSEMBLE PREDICTIONS
Predicting with model 1/5... ✅ Range: [-9.005, 9.583]
Predicting with model 2/5... ✅ Range: [-9.005, 9.583]
Predicting with model 2/5... ✅ Range: [-9.378, 9.229]
Predicting with model 3/5... ✅ Range: [-9.378, 9.229]
Predicting with model 3/5... ✅ Range: [-9.596, 10.892]
Predicting with model 4/5... ✅ Range: [-9.596, 10.892]
Predicting with model 4/5... ✅ Range: [-9.622, 9.902]
Predicting with model 5/5... ✅ Range: [-9.622, 9.902]
Predicting with model 5/5... ✅ Range: [-8.803, 9.762]

📊 ENSEMBLE STATISTICS
Num models: 5
Ensemble range: [-9.247, 9.767]
Ensemble mean:  0.014
Ensemble std:   1.461

Individual model std devs:
  Model 1: std=1.437
  Model 2: std=1.482
  Model 3: std=1.485
  Model 4: std=1.440
  Model 5: std=1.475

Prediction variance (mean): 0.009036
Prediction variance (max):  1.245033
✅ Range: [-8.803, 9.762]

📊 ENSEMBLE STATISTICS
Num models: 5
Ensemble range: [-9.247, 9.767]
Ensemble mean:  0.014
Ensemble std:   1.461

Individual model 

In [13]:
# Save ensemble predictions
y_pred = pd.DataFrame({
    "id": range(len(ensemble_preds)),
    "predicted_sideslip": ensemble_preds
})

y_pred.to_csv("predictions_multiseed_ensemble.csv", index=False)

print("\n" + "="*80)
print("✅ MULTI-SEED ENSEMBLE SAVED")
print("="*80)
print(f"File: predictions_multiseed_ensemble.csv")
print(f"Samples: {len(ensemble_preds)}")
print(f"Expected score: ~0.265-0.273° (5-8% improvement from 0.28779°)")
print("="*80)


✅ MULTI-SEED ENSEMBLE SAVED
File: predictions_multiseed_ensemble.csv
Samples: 49328
Expected score: ~0.265-0.273° (5-8% improvement from 0.28779°)
